In [113]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from tqdm import tqdm
from dataclasses import dataclass

import scipy
import sys
import sys

sys.path.append("/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/")
from MlUtlitys import PlotManager , ParamGrid

In [114]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/credit_fraud_detection/archive.zip"
windows_path = r""
df = pd.read_csv(linux_path)

In [115]:
@dataclass
class Config:
    target: str = "Class"
    test_size: float = 0.2
    seed: int = 1234
    cross_iteration: int = 3
    grid_iterations: int = 3
    verbose : int = 0
    search_iterations:int = 50
    
config = Config()

In [116]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [117]:
class DataClass():

    def __init__(self):
        
        self.x = df.drop([config.target] , axis = 1)
        self.y = df[config.target]

        self.numerical_data = self.x.select_dtypes(include = "number").columns
        self.categorical_data  = self.x.select_dtypes(exclude = "number").columns
        
data = DataClass()

In [118]:
class DataTransform(BaseEstimator , TransformerMixin):
    def fit(self , X , y = None):
        return self

    def transform(self, X , y = None):
        x = X.copy()
        x = self.NewFeatures(x)
        x = self.time_features(x)
        x = self.zscsore(x)
        x = self.drop_cols(x)
        return x
    
    def NewFeatures(self , X):
        x = X.copy()

        return x
    
    def time_features(self , X):
        x = X.copy()
        x["Hours"] = (x["Time"] // 3600) % 24
        x["Days"] = x["Time"] // (3600 * 24)
        x["IsWeekendDay"] = (x["Days"] % 7 >= 5).astype(int)
        x["NightTime"] = x["Hours"].between(0 , 4).astype(int)
       # x["Last10Transactions"] = x["Amount"].rolling(window = 10).mean() # potential dataleakage

        timr_bins = [0 , 6 , 12 , 18 , 24]
        time_labels = ["Night" , "Morning" , "Evening" , "AfterNoon"]

        x["Time_period"] = pd.cut(x["Hours"] , bins = timr_bins , labels = time_labels , right = False)

        return x
    
    def zscsore(self , X):
        x = X.copy()
        x["ZScore"] = (x["Amount"] - x["Amount"].mean()) / x["Time"].std()
        return x
    
    def drop_cols(self , X):
        x = X.copy()
        x = x.drop(["Time" , "Hours"] , axis = 1)
        return x

In [119]:
class Visualize():

    def __init__(self , OrigData):
        self.data = OrigData.copy()

    def iqr(self, TargetCol):
        q1 = self.data[TargetCol].quantile(0.25)
        q3 = self.data[TargetCol].quantile(0.75)
        iqr = q3 - q1
        return q1 , q3 , iqr
    
    def skew(self , skew_col):
        data_skew = self.data[skew_col].skew()
        print(f"Skewness of Col{data_skew}")

    def plot(self):

        for vis in self.data.numerical_data:
            fig , axes = plt.subplots(3 , 1 , figsize = (10 , 10 ) , dpi = 200)
            q1 , q3 , _ = self.iqr(TargetCol = vis)
            mean = self.data[vis].mean()
            self.skew(skew_col = vis)

            sns.histplot(data = self.data , x = vis , ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q1 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Histplot for:{vis}")

            sns.boxplot(data = self.data , X = vis , ax = axes[1])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"boxplot for:{vis}")

            sns.scatterplot(data = self.data , X = vis , y = self.data.y ,ax = axes[2])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"scatterplot for:{vis}")

            plt.tight_layout()
            plt.show()

In [120]:
class Preprocessor():

    def fit(self, X , y = None):

        self.numerical_data = X.select_dtypes(include = np.number).columns
        self.categorical_data = X.select_dtypes(exclude = np.number).columns

        self.preprocessor = ColumnTransformer([
            ("numerical_pipe" , Pipeline([
                ("numerical_imputer" , SimpleImputer(strategy = "mean")),
                ("numerical_scaler" , MinMaxScaler())
            ]),self.numerical_data),

            ("categorical_pipeline" , Pipeline([
                ("categorical_imputer" , SimpleImputer(strategy="most_frequent")),
                ("categorical_encoder" , OneHotEncoder(handle_unknown="ignore" , sparse_output=False))
            ]),self.categorical_data)
        ])
        self.preprocessor.fit(X)
        return self

    def transform(self, X):
        return self.preprocessor.transform(X)

In [121]:
def custom_pipeline(estimator , transform = False , smote = False):

    steps = []
    if transform:
        steps.append(("transformed", DataTransform()))  
    steps.append(("base_performance" , Preprocessor()))

    if smote:
        steps.append(("Smote" , SMOTE()))
        
    steps.append(("estimator", estimator))
    return Pipeline(steps) 


In [122]:
def model_varianz():
    return {

        "LogisticRegression" : LogisticRegression(random_state = config.seed),
        "DecisionTreeClassifier" : DecisionTreeClassifier(random_state = config.seed),
        "RandomForestClassifier" : RandomForestClassifier(random_state = config.seed , n_jobs = -1),
        "LinearSVC": LinearSVC(random_state = config.seed , max_iter = 1000),
        "XGBClassifier": XGBClassifier(random_state = config.seed),
        
    }

In [123]:
def split_data():
    X_train , X_test , y_train  , y_test = train_test_split(data.x,
                                                            data.y,
                                                            random_state = config.seed,
                                                            shuffle = True,
                                                            stratify = data.y,
                                                            test_size = config.test_size)
    return X_train , X_test , y_train  , y_test

In [124]:
def custom_scoring_dict():
    return {
        "roc_auc":"roc_auc",
        "f1":"f1",
        "pr_auc":"average_precision",
        }

In [125]:
def custom_data_validation(estimator , X_train , y_train , scoring_dict):
    kfold = StratifiedKFold(n_splits = 3 , shuffle = True , random_state = config.seed)
 
    return cross_validate(estimator = estimator,
                          X = X_train,
                          y = y_train,
                          return_train_score=True ,
                          return_estimator= True,
                          scoring = scoring_dict,
                          cv = kfold,
                          n_jobs= -1,
                          verbose=config.verbose,
                          )

In [126]:
def custom_grid_search(estimator ,X_train , y_train , param_grid):
    grid = RandomizedSearchCV(estimator=estimator,
                              param_distributions=param_grid,
                              return_train_score=True,
                              refit="roc_auc",
                              cv = config.cross_iteration,
                              verbose=config.verbose,
                              n_iter=config.search_iterations,
                              random_state=config.seed,
                              )
    
    grid.fit(X_train , y_train)
    return grid

In [ ]:
def plot_scores(
        self,
        data,
        x_column,
        score_columns,
        title="Model Comparison",
        ylabel="Score"
    ):
        # Falls eine Liste von Dictionaries übergeben wird
        if isinstance(data, list):
            data = pd.DataFrame(data)

        x = range(len(data))

        plt.figure(figsize=(12, 6))

        for column in score_columns:
            if column in data.columns:
                plt.plot(
                    x,
                    data[column],
                    marker="o",
                    linewidth=2,
                    label=column
                )

        plt.xticks(x, data[x_column], rotation=45)
        plt.xlabel(x_column)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
class Benchmark():

    def __init__(self , UseCV = False , UseGrid = False ):
        self.X_train , self.X_test , self.y_train , self.y_test = split_data()
        self.results = []

        self.use_cv = UseCV
        self.use_grid_search = UseGrid
        self.train()

    def train(self):

        for estimator_name , estimators in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimators , transform= False , smote=False)
            transformed_pipe = custom_pipeline(estimator=estimators , transform= True , smote=True)

            if self.use_cv:
                base_cv = custom_data_validation(estimator=base_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train,
                                                scoring_dict=custom_scoring_dict())
                
                transformed_cv = custom_data_validation(estimator=transformed_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train,
                                                scoring_dict=custom_scoring_dict())
                
                print(f"keys;{base_cv.keys()}")

                self.results.append({

                    "base_cv_train_performance":base_cv["train_roc_auc"].mean(),
                    "base_cv_test_performance":base_cv["test_roc_auc"].mean(),
                    "base_cv_std_performance":base_cv["test_roc_auc"].std(),

                    "base_cv_train_performance":base_cv["train_f1"].mean(),
                    "base_cv_test_performance":base_cv["test_f1"].mean(),
                    "base_cv_std_performance":base_cv["test_f1"].std(),

                    "base_cv_train_pr_auc_performance":base_cv["train_pr_auc"].mean(),
                    "base_cv_test_pr_auc_performance":base_cv["test_pr_auc"].mean(),
                    "base_cv_test_pr_auc_performance":base_cv["test_pr_auc"].std(),


                    "transformed_cv_train_performance":transformed_cv["train_roc_auc"].mean(),
                    "transformed_cv_test_performance":transformed_cv["test_roc_auc"].mean(),
                    "transformed_cv_std_performance":transformed_cv["test_roc_auc"].std(),

                    "transformed_cv_train_performance":transformed_cv["train_f1"].mean(),
                    "transformed_cv_test_performance":transformed_cv["test_f1"].mean(),
                    "transformed_cv_std_performance":transformed_cv["test_f1"].std(),

                    "transformed_cv_train_pr_auc_performance":transformed_cv["train_pr_auc"].mean(),
                    "transformed_cv_test_pr_auc_performance":transformed_cv["test_pr_auc"].mean(),
                    "transformed_cv_test_pr_auc_performance":transformed_cv["test_pr_auc"].std(),
                })

            if self.use_grid_search:
                base_grid = custom_grid_search(estimator=base_pipe,
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=ParamGrid().custom_paramter_grid()[estimator_name]  )

                transformed_grid = custom_grid_search(estimator=base_pipe,              
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=ParamGrid().custom_paramter_grid()[estimator_name] )
                
            self.results.append({

                "base_cv_train_performance":base_grid.best_estimator_,
                "transformed_cv_train_performance":transformed_grid.best_estimator_,
                
            })

        plot_scores(data = self.results)

In [ ]:
Benchmark(UseCV= True , UseGrid = True)

/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/p

keys;dict_keys(['fit_time', 'score_time', 'estimator', 'test_roc_auc', 'train_roc_auc', 'test_f1', 'train_f1', 'test_pr_auc', 'train_pr_auc'])


/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.clean_venv/

keys;dict_keys(['fit_time', 'score_time', 'estimator', 'test_roc_auc', 'train_roc_auc', 'test_f1', 'train_f1', 'test_pr_auc', 'train_pr_auc'])
keys;dict_keys(['fit_time', 'score_time', 'estimator', 'test_roc_auc', 'train_roc_auc', 'test_f1', 'train_f1', 'test_pr_auc', 'train_pr_auc'])
